SimpleTokenizerV2 behavior

In [ ]:
import re  # 导入正则模块，用于按标点/空白对文本进行切分

class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab  # 词表：token 字符串 -> 整数 id，直接引用外部传入的字典
        self.int_to_str={index:token for token , index in vocab.items()}  # FIX: 原代码写成 vocab.item()，dict 没有该方法会抛 AttributeError，这里改为 vocab.items()；反转词表，得到 id -> token 的映射，供解码使用
    def encode(self,text):
        # 按标点符号（,.:;?_!"()'）、"--"、空白符切分，用捕获组保留分隔符本身
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # 去掉切分产生的空字符串，并对每个 token 做 strip —— 注意这一步会把原始的空白/换行信息全部丢弃
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        # 不在词表中的 token 统一替换为特殊 token <|unk|>，避免后续查表报错
        preprocessed=[
            item if item in self.str_to_int
            else '<|unk|>' for item in preprocessed
        ]
        # 将 token 列表转换成对应的整数 id 列表
        return [self.str_to_int[item] for item in preprocessed]
    def decode(self,ids):
        # 用空格把所有 id 对应的 token 拼接成一个字符串
        text = ' '.join(self.int_to_str[index] for index in ids)
        # 用正则去掉标点符号前面多余的空格，例如 "tea ." -> "tea."
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)

In [ ]:
training_text = 'I, HAD tea.\nI-- HAD tea.'  # 用于构建词表的训练文本
# 用与 V2 encode 相同的规则切分训练文本，得到所有候选 token（标点、单词等）
v2_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', training_text)
# 去掉空字符串，并对每个 token 做 strip，因此空白符本身不会成为词表条目
v2_tokens = [item.strip() for item in v2_tokens if item.strip()]
# 去重、排序后再加上特殊 token <|unk|>，构建 token -> id 的词表
v2_vocab = {token: index for index, token in enumerate(sorted(set(v2_tokens)) + ['<|unk|>'])}
tokenizer_v2 = SimpleTokenizerV2(v2_vocab)  # 用上面构建的词表实例化 V2 分词器

sample_text = 'I,  HAD tea.\nI-- HAD tea.'  # 注意 "I," 后面是两个空格，与训练文本的间距不同
decoded_text = tokenizer_v2.decode(tokenizer_v2.encode(sample_text))
# 因为 V2 在 encode 阶段丢弃了空白信息，decode 时只能用单个空格重新拼接，
# 所以还原出来的文本与原文在空白/换行上必然不一致
assert decoded_text != sample_text

print('Original:', repr(sample_text))
print('Decoded: ', repr(decoded_text))

SimpleTokenizerV3 behavior

In [ ]:
class SimpleTokenizerV3:
    def __init__(self, vocab):
        self.str_to_int = vocab  # 词表：token -> id
        self.int_to_str = {index: token for token, index in vocab.items()}  # 反转词表，得到 id -> token，用于解码

    def encode(self, text):
        # 切分规则与 V2 相同，但不再对 token 做 strip，因此空白符（空格、换行等）也会被保留为独立 token
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # 只过滤掉切分产生的空字符串，空白 token 本身予以保留
        preprocessed = [item for item in preprocessed if item != '']
        # 用 dict.get 查表，词表外的 token 统一映射为 <|unk|> 对应的 id，避免 KeyError
        return [self.str_to_int.get(item, self.str_to_int['<|unk|>']) for item in preprocessed]

    def decode(self, ids):
        # 因为 encode 阶段保留了所有空白 token，这里可以直接拼接（不加分隔符），从而精确还原原始文本
        return ''.join(self.int_to_str[index] for index in ids)

In [ ]:
# 用与 V3 encode 相同的规则切分训练文本
v3_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', training_text)
# 词表同样包含空白类 token（如空格、换行），只排除空字符串；再加上 <|unk|> 特殊 token
v3_vocab = {token: index for index, token in enumerate(sorted(set(token for token in v3_tokens if token)) + ['<|unk|>'])}
tokenizer_v3 = SimpleTokenizerV3(v3_vocab)
decoded_text = tokenizer_v3.decode(tokenizer_v3.encode(sample_text))
# 因为空白信息也被编码进了 token 序列，V3 可以做到完全无损的编码-解码往返
assert decoded_text == sample_text

print('Original:', repr(sample_text))
print('Decoded: ', repr(decoded_text))

# 额外测试几种不同的空白/分隔场景，验证 V3 在这些情况下依然能精确还原原文
for sample in ('I-- HAD', 'I, HAD', 'I  HAD', 'I\nHAD'):
    assert tokenizer_v3.decode(tokenizer_v3.encode(sample)) == sample

# 验证未登录词（不在词表中的词）会被正确映射为 <|unk|> 对应的 id
assert tokenizer_v3.str_to_int['<|unk|>'] in tokenizer_v3.encode('unknownword')